# 🎯 AI Resume Analyzer - Complete Project Walkthrough

- Show the end-to-end flow: PDFs -> outputs/
- Highlight this is a live coding demo
- Keep code cells empty for audience participation

### Setup: imports and API key

- Use .env and load_dotenv()
- OPENAI_API_KEY must be set
- Fail fast if key is missing

## 📝 Section 1: Environment Setup

- Use .env and load_dotenv()
- OPENAI_API_KEY must be set
- Fail fast if key is missing

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("OPENAI_API_KEY is loaded")

OPENAI_API_KEY is loaded


### 2. Imports for LangChain & helpers

- ChatOpenAI for LLM calls
- PyPDFLoader to read PDFs
- PromptTemplate + PydanticOutputParser for structure

## 📝 Section 2: LangChain Imports

- ChatOpenAI for LLM calls
- PyPDFLoader to read PDFs
- PromptTemplate + PydanticOutputParser for structure

#### Pydantic Output Parser:
- Force the LLMs to return structured JSON matching to our schema
- Validate output format automatically
- Whenever there is inconsistent responses, pydantic reduces the errors for prod 

1. ChatOpenAI -> System Message/ Human Message/ AI Messaage
2. OpenAI -> string as input and string as output

In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
import json

### 3. Load Resume PDF

- Resumes folder should contain PDFs
- List comprehension filters .pdf case-insensitive
- Collect docs for each file

In [ ]:
Docx2txtLoader
tqdm -> Process large batches ar once


In [ ]:
for fil

In [ ]:
resume_folder = "resumes"

resume_files = [
    f for f in os.listdir(resume_folder)
    if f.lower().endswith(".pdf")
]

print("Found Resumes:", resume_files)


all_docs = []

for file in resume_files:
    path = os.path.join(resume_folder, file) 
    print(f"\nLoading: {file}")
    
    loader = PyPDFLoader(path) # Create a loader instance
    docs = loader.load() # Return list of Docuemnt objects
    
    all_docs.append({
        "filename": file,
        "docs": docs
    })

print("\nCompleted loading all resumes")

Found Resumes: ['Alex Johnson.pdf', 'Arjun Nair.pdf']

Loading: Alex Johnson.pdf

Loading: Arjun Nair.pdf

Completed loading all resumes


## 📝 Section 3: Loading Resume PDFs

- Resumes folder should contain PDFs
- List comprehension filters .pdf case-insensitive
- Collect docs for each file

### Helper Function to Combine Pages & Clean Text

- Join page_content with blank lines
- Regex collapses whitespace
- Cleaner text lowers token cost

## 📝 Section 4: Text Cleaning

- Join page_content with blank lines
- Regex collapses whitespace
- Cleaner text lowers token cost

In [9]:
for d in docs:
    print(d)

page_content='Arjun Nair
Email: user7@email.com | Phone: +91-9876500007 | LinkedIn: linkedin.com/in/user7
Professional Summary
Cloud engineer with AWS expertise, focusing on cost optimization and scalability.
Skills
Programming Languages:
Python, R, SQL
Frameworks:
TensorFlow, Scikit-learn
Tools:
Linux, GCP, Git
Other:
Machine Learning, Statistics
Certifications
- Oracle Certified Java Programmer
- Udemy: Python for Data Science
Professional Experience
Data Analyst – Insight Analytics (2021 – Present)
 Performed data cleaning and EDA.
 Built dashboards in Power BI.
 Provided insights to stakeholders.
Junior Developer – InnovateX Labs (2020 – 2022)
 Worked on frontend development.
 Improved SQL queries performance.
 Assisted in chatbot training.
Education
B.Tech in Computer Science, Anna University (2017 – 2021), CGPA: 8.4/10' metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-06T14:37:33+00:00', 'author': '(anon

Resumes often has:
- Multiple spaces, tab, new line


"\n\n".join([d.page_content for d in docs]):

1. Combine all the pages into one string, 
2. Use \n\n (double newline) -> to preserve new paragraph structure
3. List Comprehension extracts text from each Docuemtn object 

re.sub(r'\s+', " ", text).strip():

- Regular Expression:
- \s+ -> Matches one or more whitespace characters (space, tab, newlines)
- Replace them with a single space
- `.strip()` remove leading/ trailing whitespace

In [10]:
import re

def combine_and_clean(docs):
    text = "\n\n".join([d.page_content for d in docs])
    text = re.sub(r'\s+', " ", text).strip()
    return text

### Initialize LLM

- Model: gpt-4o-mini
- temperature=0 for deterministic extraction
- Reuse one client instance

## 📝 Section 5: Initialize LLM

- Model: gpt-4o-mini
- temperature=0 for deterministic extraction
- Reuse one client instance

In [11]:
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    api_key = OPENAI_API_KEY
)

### Pydantic Schema for structured JSON extraction

- Optional fields using str | None
- Use Field(default_factory=list) for lists
- Education is a nested list schema

## 📝 Section 6: Pydantic Schema ⭐ CRITICAL CONCEPT

- Optional fields using str | None
- Use Field(default_factory=list) for lists
- Education is a nested list schema

## 📝 Creating the Parser

- PydanticOutputParser enforces schema
- Generates format instructions automatically
- parse() validates LLM output

### Extraction Prompt Template + Chain

- Prompt asks for JSON only
- Pipe syntax: prompt | llm
- Input variable: resume_text

## 📝 Section 7: Extraction Chain ⭐ LCEL INTRODUCTION

- Prompt asks for JSON only
- Pipe syntax: prompt | llm
- Input variable: resume_text

### Skill Gap Prompt + Pipeline (Updated)

- Inputs: candidate_skills, required_skills
- Return missing_skills and recommendations
- Same pipe pattern: prompt | llm

## 📝 Section 8: Skill Gap Analysis Chain

- Inputs: candidate_skills, required_skills
- Return missing_skills and recommendations
- Same pipe pattern: prompt | llm

### Final Report Prompt + Pipeline

- Combine candidate_json + skill_gap_json
- Ask for score 0-100 and short recommendation
- Output JSON only

## 📝 Section 9: Final Report Chain

- Combine candidate_json + skill_gap_json
- Ask for score 0-100 and short recommendation
- Output JSON only

### Job Description & Skill Extraction

- clean_json_text strips code fences
- JD prompt returns list of skills
- Fallback skills list on parse errors

## 📝 Section 10: Job Description Processing

- clean_json_text strips code fences
- JD prompt returns list of skills
- Fallback skills list on parse errors

### Pipeline Function (Modern Invocation)

- Orchestrates extract -> skill gap -> final
- Guard each step with try/except
- Return final_report, parsed, skill_gap_json

## 📝 Section 11: The Main Pipeline Function ⭐ ORCHESTRATION

- Orchestrates extract -> skill gap -> final
- Guard each step with try/except
- Return final_report, parsed, skill_gap_json

### Process ALL Resumes + Save Outputs

- Iterate all_docs
- slugify filenames to save outputs
- Write each result to outputs/<name>.json

## 📝 Section 12: Batch Processing

- Iterate all_docs
- slugify filenames to save outputs
- Write each result to outputs/<name>.json

### View Results Summary

- Load outputs/*.json
- Jaccard similarity on skills
- Rank with pandas dataframe

## 📝 Section 13: Ranking & Analysis ⭐ DATA SCIENCE

- Load outputs/*.json
- Jaccard similarity on skills
- Rank with pandas dataframe

### Export to CSV

- Save df to outputs/candidate_ranking.csv
- index=False for clean file
- Share with HR or import elsewhere

## 📝 Section 14: Export to CSV

- Save df to outputs/candidate_ranking.csv
- index=False for clean file
- Share with HR or import elsewhere